# Datenquelle Detroit: RMS Crime Incidents

**Zweck:** Dieses Notebook dokumentiert die Detroit-Quelle, bevor Daten geladen
oder transformiert werden. Es beschreibt, woher die Daten stammen, wie sie
technisch bereitgestellt werden und was eine Zeile fachlich bedeutet.

Es ist das Gegenstück zu [`01_chicago_source.ipynb`](01_chicago_source.ipynb).
Der Aufbau ist derselbe, damit sich die beiden Quellen Abschnitt für Abschnitt
vergleichen lassen — denn genau darauf läuft das Projekt hinaus.

**Es behauptet so wenig wie möglich.** Jede Zahl in diesem Notebook wird von
einer Zelle gerechnet, entweder gegen die Schnittstelle oder gegen den
Bronze-Bestand. Wo eine Zelle Netz braucht, steht es dabei.

**Stand der Dokumentation:** 11. September 2026

## 1. Kurzüberblick

| Merkmal | Beschreibung |
|---|---|
| Datensatz | **RMS Crime Incidents** |
| Herausgeber | City of Detroit, Open Data Portal |
| Datenverantwortlicher | Detroit Police Department (DPD) |
| Quellsystem | RMS — Records Management System |
| Plattform | ArcGIS Online (Esri), gehostet bei `services2.arcgis.com` |
| Schnittstelle | ArcGIS REST, Feature Service, Layer `0` |
| Layer-Name | `rms_crime_incidents` |
| Zeitraum | ab 2016 belastbar, einzelne Altfälle deutlich früher |
| Aktualisierung | täglich |
| Räumlicher Bezug | City of Detroit, Punktgeometrie WGS 84 |
| Deliktschlüssel | **MICR** — Michigan Incident Crime Reporting |

**Der Unterschied zu Chicago in einem Satz:** Chicago liefert eine
Socrata-Tabelle mit SQL-ähnlicher Abfragesprache und eigenem Deliktschlüssel
(IUCR); Detroit liefert einen ArcGIS-Feature-Layer mit Geometrie und dem
Landesschlüssel MICR. Beide zählen dasselbe, aber keine Spalte heisst gleich.

## 2. Um welche Art von Quelle handelt es sich?

Das Detroit Police Department führt seine Fälle im **RMS**. Von dort läuft eine
Aufbereitungsstrecke in ein Data Warehouse und weiter in einen
**ArcGIS-Feature-Layer**, den die Stadt öffentlich hostet.

Das ist kein Detail am Rande: Die Feldbeschreibung des Layers nennt die
Herkunftstabelle im Warehouse offen, und daran lässt sich ablesen, dass wir
**nicht** das RMS selbst sehen, sondern ein aufbereitetes Mart-Modell. Die
nächste Zelle liest das aus den Metadaten.

Ein Feature Layer ist fachlich eine Tabelle, deren Zeilen zusätzlich eine
**Geometrie** tragen — hier je Fall ein Punkt. Abgefragt wird über die
**ArcGIS REST API**: kein SQL, sondern ein `where`-Ausdruck, eine Feldauswahl
und ein Ausgabeformat als HTTP-Parameter.

In [ ]:
import json
import sys
import urllib.parse
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Projekt-Root mit config/ nicht gefunden')

# Die Pipelines liegen unter src/crime/ und heissen von dort aus
# detroit_pipeline - so wie pyproject.toml sie auch fuer pytest bekannt macht.
_pakete = str(PROJECT_ROOT / 'src' / 'crime')
if _pakete not in sys.path:
    sys.path.insert(0, _pakete)

LAYER = ('https://services2.arcgis.com/qvkbeam7Wirps6zC/arcgis/rest/services/'
         'RMS_Crime_Incidents/FeatureServer/0')

# Die Adresse steht hier nochmals ausgeschrieben, damit das Notebook ohne das
# Paket lesbar bleibt - und wird sofort dagegen geprueft.
from detroit_pipeline.detroit_arcgis import DETROIT_LAYER_URL  # noqa: E402
assert LAYER == DETROIT_LAYER_URL, 'Notebook und Pipeline zeigen woanders hin'


def arcgis(pfad='', **parameter):
    """Ein GET gegen den Layer. Gibt die geparste Antwort zurueck."""
    parameter.setdefault('f', 'json')
    ziel = f'{LAYER}{pfad}?{urllib.parse.urlencode(parameter)}'
    with urllib.request.urlopen(ziel, timeout=120) as antwort:
        return json.load(antwort)


try:
    meta = arcgis()
    NETZ = True
except Exception as fehler:                      # noqa: BLE001 - Diagnose genuegt
    meta, NETZ = {}, False
    print('Layer nicht erreichbar:', fehler)

if NETZ:
    print('Layer-Name        :', meta['name'])
    print('Herkunftstabelle  :', meta['description'])
    print('Geometrie         :', meta['geometryType'])
    print('Objekt-ID-Feld    :', meta['objectIdField'])
    print('max. Zeilen/Abruf :', meta['maxRecordCount'])
    print('Faehigkeiten      :', meta['capabilities'])
    print('Felder            :', len(meta['fields']))

**`capabilities` ist `Query`** — der Layer lässt sich nur lesen. Das ist die
erwartete Einstellung für eine offene Quelle, aber es lohnt sich, es geprüft zu
haben statt angenommen.

**`maxRecordCount` ist 2000.** Mehr Zeilen gibt der Dienst pro Abruf nicht
heraus, egal was man anfragt. Ein Jahrgang Detroit sind rund 88'000 Fälle;
gepaged werden muss also in jedem Fall (Abschnitt 5).

## 3. Was eine Zeile bedeutet

Eine Zeile ist **ein Delikt innerhalb eines Vorfalls**, nicht der Vorfall
selbst. Ein Einbruch mit anschliessender Körperverletzung erzeugt zwei Zeilen
mit derselben `case_id` und verschiedenen `incident_entry_id`.

| Feld | Bedeutung |
|---|---|
| `incident_entry_id` | Schlüssel der Zeile — ein Delikt in einem Vorfall |
| `case_id`, `crime_id` | Schlüssel des Vorfalls, mehrere Zeilen teilen ihn |
| `report_number` | Aktenzeichen des DPD |

Das ist **nicht dieselbe Granularität wie bei Chicago** — auch wenn diese Zelle
bis zum 13.9.2026 das Gegenteil behauptete. Chicago führt je Vorfall **ein**
Delikt: 259'272 Zeilen auf 259'243 Fälle im Jahrgang 2024, und die einzigen
Mehrfachzeilen sind Mordopfer (ein Datensatz je Opfer, 25 Fälle). Detroit
führt je Vorfall **jedes** Delikt: 85'345 Taten auf 80'229 Fälle, 4'658 Fälle
mit zwei bis vier Taten. Auf Tatebene zählt Detroit damit 6,4 % mehr Taten als
Fälle — Delikte, die Chicago nie zählt. Silver gleicht das mit `counts_case`
an: je Fall die Zeile mit dem schwersten Delikt, für beide Städte nach
derselben Rangfolge (`FALLKORN_RANG` in `build_crime_offense`). Die nächste
Zelle misst, wie viele Vorfälle mehr als ein Delikt tragen.

In [ ]:
import glob

import duckdb

_laeufe = sorted(glob.glob(
    str(PROJECT_ROOT / 'data' / 'bronze' / 'city=detroit' / '**' / 'run_*' / 'part-0001.parquet'),
    recursive=True))
BRONZE = f'read_parquet({_laeufe})' if _laeufe else None
con = duckdb.connect()

if BRONZE is None:
    print('Kein Detroit-Bronze-Lauf vorhanden. Zuerst:')
    print('  python -m detroit_pipeline.extract_detroit --mode delta --years 2024')
else:
    zeilen, faelle, mehrfach = con.execute(f'''
        WITH je_fall AS (
            SELECT attributes.case_id AS fall, count(*) AS delikte
            FROM {BRONZE} GROUP BY 1
        )
        SELECT (SELECT count(*) FROM {BRONZE}),
               count(*),
               count(*) FILTER (WHERE delikte > 1)
        FROM je_fall
    ''').fetchone()
    print(f'Zeilen (Delikte)          : {zeilen:,}')
    print(f'Vorfaelle (case_id)       : {faelle:,}')
    print(f'davon mit >1 Delikt       : {mehrfach:,} = {mehrfach / faelle:.1%}')
    print(f'Delikte je Vorfall im Mittel: {zeilen / faelle:.2f}')

## 4. Die 28 Quellspalten

Die nächste Zelle listet das Schema, wie der Dienst es meldet — nicht wie wir
es in Erinnerung haben. Die Spalte *in Bronze belegt* rechnet dazu, wie viele
Zeilen des Jahrgangs 2024 tatsächlich einen Wert tragen. Ein Feld, das im
Schema steht und in den Daten leer bleibt, ist für die Auswertung nicht
vorhanden.

In [ ]:
if NETZ:
    felder = [(f['name'], f['type'].replace('esriFieldType', ''), f.get('alias') or '')
              for f in meta['fields']]

    belegt = {}
    if BRONZE:
        ausdruck = ', '.join(
            f"count(attributes.{n}) FILTER (WHERE trim(CAST(attributes.{n} AS VARCHAR)) <> '')"
            for n, _t, _a in felder if n != 'ESRI_OID')
        werte = con.execute(f'SELECT {ausdruck} FROM {BRONZE}').fetchone()
        gesamt = con.execute(f'SELECT count(*) FROM {BRONZE}').fetchone()[0]
        belegt = dict(zip([n for n, _t, _a in felder if n != 'ESRI_OID'], werte))

    print(f"{'Feld':28}{'Typ':10}{'in Bronze belegt':>18}   Bezeichnung im Dienst")
    for name, typ, alias in felder:
        anteil = ''
        if name in belegt and gesamt:
            anteil = f'{belegt[name] / gesamt:>17.1%}'
        print(f'{name:28}{typ:10}{anteil:>18}   {alias[:34]}')

**Drei Gruppen fallen auf.**

*Abgeleitete Zeitfelder.* `incident_time`, `incident_day_of_week`,
`incident_hour_of_day` und `incident_year` sind alle aus
`incident_occurred_at` gerechnet. Sie sind bequem und **gefährlich**: Sie
tragen keine Zeitzone, während das Datumsfeld selbst in UTC deklariert ist
(Abschnitt 6). Die Pipeline benutzt ausschliesslich `incident_occurred_at`.

*Räumliche Zuordnungen aus der Quelle.* `census_block_2020_geoid`,
`neighborhood`, `council_district`, `police_precinct`, `zip_code`. Wir nehmen
sie **nicht** als Raumeinheit, sondern verschneiden `longitude`/`latitude`
selbst gegen die TIGER/Line-Tracts — sonst wäre Detroit anders verortet als
Chicago, wo es keine solche Spalte gibt.

*Zwei Deliktschlüssel nebeneinander.* `state_offense_code` und
`arrest_charge`. Was darin wirklich steht, ist Abschnitt 7.

## 5. Technischer Zugriff

### Adressen

- Layer: `…/RMS_Crime_Incidents/FeatureServer/0`
- Metadaten: derselbe Pfad mit `?f=json`
- Abfrage: derselbe Pfad mit `/query`

### Kein Schlüssel

Der Dienst ist **ohne Token** erreichbar. Chicago braucht dagegen einen
Socrata App Token aus `.env`. Für Detroit gibt es entsprechend nichts
geheimzuhalten — und nichts, was in einem Manifest landen könnte.

### Seitenweise über Object-IDs, nicht über Offsets

Der Layer meldet **kein** `supportsPagination`. Ein `resultOffset` ist damit
nicht verlässlich: Ändert sich die Sortierung zwischen zwei Abrufen, entstehen
Lücken oder Dubletten, ohne dass irgendetwas fehlschlägt.

Die Pipeline holt deshalb erst **alle Object-IDs** des Zeitfensters mit
`returnIdsOnly=true` — das liefert der Dienst in einem Zug und ohne
`maxRecordCount`-Grenze — und lädt die Zeilen danach in Blöcken über
`objectIds`. Die Menge steht damit vor dem ersten Datenabruf fest und ist
nachher prüfbar.

Die Blöcke gehen bewusst **als POST** hinaus: Eine Liste von 2000 IDs sprengt
die URL-Längenlimits mancher Proxies.

In [ ]:
if NETZ:
    gesamt_dienst = arcgis('/query', where='1=1', returnCountOnly='true')['count']
    spanne = arcgis('/query', where='1=1', outStatistics=json.dumps([
        {'statisticType': 'min', 'onStatisticField': 'incident_occurred_at',
         'outStatisticFieldName': 'frueheste'},
        {'statisticType': 'max', 'onStatisticField': 'incident_occurred_at',
         'outStatisticFieldName': 'spaeteste'}]))['features'][0]['attributes']

    from datetime import datetime, timezone
    def utc(ms):
        return datetime.fromtimestamp(ms / 1000, timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

    print(f'Zeilen im Dienst insgesamt: {gesamt_dienst:,}')
    print(f'frueheste incident_occurred_at: {utc(spanne["frueheste"])}')
    print(f'spaeteste incident_occurred_at: {utc(spanne["spaeteste"])}')
    print(f'-> bei {meta["maxRecordCount"]} Zeilen je Abruf waeren das mindestens '
          f'{-(-gesamt_dienst // meta["maxRecordCount"]):,} Abrufe fuer den ganzen Bestand')

**Die früheste Zeile liegt weit vor jeder plausiblen Erfassung.** Das ist kein
Fehler der Schnittstelle, sondern ein Altlast-Wert in den Daten — und ein
Argument dafür, den Jahrgang beim Laden hart zu begrenzen, statt „alles" zu
holen und später zu filtern.

## 6. Die Zeitzonenfalle

**Der Layer deklariert seine Datumsfelder ausdrücklich als UTC.** Die nächste
Zelle liest die Deklaration aus den Metadaten.

Das ist der wichtigste einzelne Satz in diesem Notebook. Wer ein Jahresfenster
mit `TIMESTAMP '2024-01-01 00:00:00'` zieht und dabei den 1. Januar **in
Detroit** meint, bekommt in Wahrheit den 31. Dezember 2023 um 19:00 Ortszeit —
und damit Fälle aus dem Vorjahr in einen Jahrgang 2024.

Genau das ist in diesem Projekt passiert und wurde an beiden Beständen
nachgemessen (Konzept 8.6). Die Quelle hat es die ganze Zeit gesagt; gelesen
hat es niemand.

In [ ]:
if NETZ:
    print('dateFieldsTimeReference:',
          json.dumps(meta.get('dateFieldsTimeReference'), ensure_ascii=False))

    from datetime import date

    from detroit_pipeline.detroit_arcgis import build_local_date_window, build_year_where

    # Genau die Funktionen, die auch die Extraktion aufruft - nicht nachgebaut.
    fenster = build_local_date_window(date(2024, 1, 1), date(2025, 1, 1))
    JAHR_2024 = build_year_where(2024)
    print()
    print('So zieht die Pipeline den Jahrgang 2024:')
    print('  Ortszeit Detroit:', fenster.start_local, 'bis', fenster.end_local)
    print('  daraus in UTC   :', fenster.start_utc, 'bis', fenster.end_utc)
    print('  WHERE           :', JAHR_2024)

    naiv = arcgis('/query', returnCountOnly='true', where=(
        "incident_occurred_at >= TIMESTAMP '2024-01-01 00:00:00' AND "
        "incident_occurred_at <  TIMESTAMP '2025-01-01 00:00:00'"))['count']
    richtig = arcgis('/query', returnCountOnly='true', where=JAHR_2024)['count']
    print()
    print(f'Zeilen mit naivem Fenster (Ortszeit als UTC gelesen): {naiv:,}')
    print(f'Zeilen mit korrektem Fenster                       : {richtig:,}')
    print(f'Unterschied                                        : {naiv - richtig:+,}')

**Der Unterschied ist klein — und das ist die Gefahr.** Eine Abweichung von
einem knappen Dutzend Fällen auf 88'000 fällt in keiner Plausibilitätsprüfung
auf.

**Der Nettowert untertreibt sogar noch.** Das falsche Fenster verschiebt *beide*
Grenzen um fünf Stunden: Es nimmt Fälle vom Silvesterabend des Vorjahres auf und
verliert dieselbe Zahl am Jahresende. Am alten und neuen Bestand gemessen waren
das 66 Fälle herein und 56 hinaus — netto zehn, tatsächlich 122 falsch
einsortierte Zeilen (Konzept 8.6).
Sie fällt nur auf, wenn man gezielt fragt, ob ein Jahrgang 2024 Fälle aus 2023
enthält. Genau diese Frage läuft als Test gegen die gebaute Silver-Tabelle.

`DetroitDateWindow` löst die Sommerzeit dabei mit `zoneinfo` auf: Der 1. Januar
liegt in `EST` (UTC−5), der 1. Juni in `EDT` (UTC−4). Ein fest verdrahteter
Versatz wäre die halbe Jahreshälfte falsch.

## 7. Zwei Deliktfelder, und keines heisst, wie es klingt

Detroit liefert zwei Schlüsselfelder. Beide gehören zu **MICR**, dem
Meldeschlüssel des Bundesstaats Michigan — aber sie sind nicht dasselbe, und
die Feldnamen führen in die Irre.

Die nächste Zelle rechnet nach, was tatsächlich drinsteht.

In [ ]:
if BRONZE:
    import csv as _csv
    micr = {z['micr_code']: z for z in _csv.DictReader(
        (PROJECT_ROOT / 'config' / 'micr_nibrs_mapping.csv').open(encoding='utf-8'))}

    for feld in ('state_offense_code', 'arrest_charge'):
        laengen = con.execute(f'''
            SELECT length(trim(attributes.{feld})) AS n, count(*) AS c
            FROM {BRONZE} GROUP BY 1 ORDER BY 1''').fetchall()
        verschieden = con.execute(f'''
            SELECT count(DISTINCT trim(attributes.{feld})) FROM {BRONZE}''').fetchone()[0]
        print(f'{feld:20} {verschieden:>4} verschiedene Werte, Laengen '
              + ', '.join(f'{n}-stellig: {c:,}' for n, c in laengen))

    arten = con.execute(f'''
        SELECT trim(attributes.state_offense_code) AS code, count(*) AS n
        FROM {BRONZE} GROUP BY 1''').fetchall()
    zaehler = {'file_class': 0, 'arrest_charge': 0, 'unbekannt': 0}
    for code, n in arten:
        zaehler[micr.get(code, {}).get('micr_code_type', 'unbekannt')] += n
    print()
    print('Was in state_offense_code steckt, nach unserer MICR-Tabelle:')
    for art, n in zaehler.items():
        print(f'  {art:16} {n:>8,} Faelle')

**`state_offense_code` ist vierstellig und mischt zwei Codelisten.** MICR führt
**File Classes** — was passiert ist — und **Arrest Charges** — wessen jemand
beschuldigt wird. Beide erscheinen in demselben Feld. Der Crosswalk
`config/micr_nibrs_mapping.csv` hält deshalb je Zeile fest, welche Art vorliegt;
Notebook [`32_nibrs_source.ipynb`](32_nibrs_source.ipynb) führt das aus.

**`arrest_charge` ist fünfstellig** und trägt trotz seines Namens die
**File Class in voller Länge** — also das, was unsere Tabelle in der Spalte
`micr_file_class` führt. Die nächste Zelle prüft diese Lesart, statt sie zu
behaupten.

In [ ]:
if BRONZE:
    paare = con.execute(f'''
        SELECT trim(attributes.state_offense_code) AS kurz,
               trim(attributes.arrest_charge)      AS lang,
               count(*) AS n
        FROM {BRONZE} GROUP BY 1, 2''').fetchall()

    gleich = anders = unbekannt = 0
    beispiele = []
    for kurz, lang, n in paare:
        zeile = micr.get(kurz)
        if zeile is None:
            unbekannt += n
        elif zeile['micr_file_class'] == lang:
            gleich += n
        else:
            anders += n
            beispiele.append((n, kurz, lang, zeile['micr_file_class'],
                              zeile['micr_description'][:32]))

    summe = gleich + anders + unbekannt
    print('Deckt sich Detroits arrest_charge mit unserer Spalte micr_file_class?')
    print(f'  ja            : {gleich:>7,} = {gleich / summe:.1%}')
    print(f'  nein          : {anders:>7,} = {anders / summe:.1%}')
    print(f'  Code unbekannt: {unbekannt:>7,}')
    print()
    print('Die groessten Abweichungen:')
    for n, kurz, lang, unser, bez in sorted(beispiele, reverse=True)[:6]:
        print(f'  {kurz}  Detroit {lang}  wir {unser}  {bez:<34}{n:>6,} Faelle')

**Die Lesart trägt für über 97 % der Zeilen.** Der Rest ist kein Widerspruch
zur Deutung, sondern eine offene Frage an die Quelle: Detroit setzt dort eine
andere File Class, als das Handbuch dem vierstelligen Code zuordnet.

**Für die Pipeline ändert das nichts** — der Crosswalk hängt an
`state_offense_code`, nicht an diesem Feld. Es ist hier festgehalten, damit
niemand es später für einen bequemeren Schlüssel hält, ohne die 2,8 % zu kennen.

**Und `offense_category`?** Detroit liefert zusätzlich eine eigene, grobe
Gliederung in rund 30 Kategorien. Sie ist nicht MICR, nicht NIBRS und hat kein
Gegenstück in Chicago. Sie bleibt deshalb ungenutzt.

### 7.1 Ein Fund: das ungenutzte Feld beantwortet eine offene Frage

Unter den grössten Abweichungen oben stehen **`3072`, `5421` und `5422`** — genau
die drei Codes, die der MICR-Crosswalk bis heute **gar nicht** zuordnet. Sie sind
mit rund 1'650 Fällen der grösste unzugeordnete Posten des ganzen Projekts. Die
Notiz dazu lautet seit dem Aufbau: *Detroits Beschreibung widerspricht dem
MICR-Handbuch; das kann nur Detroit auflösen.*

Detroit löst es — im Feld, das wir nicht benutzen. Die nächste Zelle stellt
gegenüber, was das vierstellige und was das fünfstellige Feld zu diesen drei
Codes sagt.

In [ ]:
if BRONZE:
    nach_file_class = {z['micr_file_class']: z for z in micr.values() if z['micr_file_class']}

    for code in ('3072', '5421', '5422'):
        zeile = micr[code]
        treffer = con.execute(f'''
            SELECT trim(attributes.arrest_charge)     AS lang,
                   trim(attributes.charge_description) AS text,
                   count(*)                            AS n
            FROM {BRONZE}
            WHERE trim(attributes.state_offense_code) = '{code}'
            GROUP BY 1, 2 ORDER BY n DESC''').fetchall()
        anzahl = sum(n for _l, _t, n in treffer)

        print(f'{code}  {anzahl:>5,} Faelle')
        print(f'      unser Crosswalk  : file_class {zeile["micr_file_class"] or "(leer)"}, '
              f'NIBRS {zeile["nibrs_code"] or "(leer)"}')
        for lang, text, n in treffer:
            ziel = nach_file_class.get(lang)
            print(f'      Detroit sagt     : arrest_charge {lang}, charge_description '
                  f'"{text}"  ({n:,} Faelle)')
            if ziel:
                print(f'      das ist bei uns  : File Class {ziel["micr_code"]} '
                      f'"{ziel["micr_description"]}" -> NIBRS {ziel["nibrs_code"]}')
            else:
                print('      das ist bei uns  : unbekannt')
        print()

**Detroit widerspricht dem Handbuch — aber nicht sich selbst.** Zu jedem der
drei Codes sagen *zwei* Felder dasselbe, und zwar in jeder einzelnen Zeile:

| | `state_offense_code` | Handbuch dazu | Detroits `arrest_charge` | Detroits `charge_description` |
|---|---|---|---|---|
| | `3072` | *Retail Fraud Misrepresentation 2nd*, File Class `30001` | `30002` | *Retail Fraud – Theft* |
| | `5421`, `5422` | *Operating ORV Under the Influence*, File Class `54002` | `54001` | *Accident, Hit & Run* |

Die File Class, die Detroit mitliefert, trägt im Handbuch **genau den Namen**,
den Detroit in `charge_description` schreibt. Beide Felder sind über alle 1'647
Zeilen eindeutig — es gibt keine zweite Lesart, die sie stützen würde.

**Warum die Vorsicht trotzdem berechtigt war.** Die Notiz im Crosswalk entstand
aus dem Handbuch allein, und dort ist `3072` nun einmal *Misrepresentation* —
das wäre `26A False Pretenses` statt `23C Shoplifting`, ein anderes Delikt. Erst
das fünfstellige Feld, das wir bis zu diesem Notebook nicht gelesen haben, zeigt,
dass Detroit die Nummer anders vergibt als das Handbuch sie führt.

**Ein Vorbehalt bleibt stehen.** `offense_description` weicht bei 40 der 1'447
Zeilen von `charge_description` ab — dort steht die Anklage *Retail Fraud* in
einem Vorfall, den Detroit als Körperverletzung oder Sachbeschädigung führt. Das
ist kein Widerspruch, sondern die Granularität der Quelle: Eine Zeile ist eine
Anklage, nicht der Vorfall. Der Crosswalk bildet Anklagen ab, und das bleibt so.

**Der Crosswalk ist inzwischen umgebucht.** `3072` steht auf `23C Shoplifting`,
`5421` und `5422` auf `90Z` — mit `detroit_charge_file_class` als Herkunft, weil
der Beleg diese Rechnung ist und nicht das Handbuch. Detroits Abdeckung steigt
damit von 98,10 % auf **100 %**, und Detroits Shoplifting-Zahl von 1'740 auf
3'187. Drei Tests halten fest, dass genau diese drei Zeilen vom Handbuch
abweichen und ihren Beleg mitführen.

## 8. Datenqualität, Datenschutz und Grenzen

**Personenbezug.** Die Quelle enthält keine Namen, keine Adressen und keine
Personenmerkmale. Der Ortsbezug ist die nächstgelegene Kreuzung plus ein
Koordinatenpaar — bewusst nicht die Hausnummer. Chicago macht es mit dem
`block`-Feld genauso.

**Was die Quelle nicht sagt.** Sie zeigt **gemeldete** Delikte. Was nicht
angezeigt wird, steht nicht darin, und die Anzeigebereitschaft ist weder über
die Zeit noch zwischen zwei Städten konstant. Jeder Vergleich Chicago–Detroit
ist damit ein Vergleich der *Meldungen*, nicht der Taten.

**Rückwirkende Änderungen.** `updated_at` ändert sich, wenn ein Fall
nachbearbeitet wird — auch Jahre später. Ein einmal geladener Jahrgang ist
deshalb kein endgültiger Stand. Die nächste Zelle misst, wie weit die
Nachbearbeitung im Jahrgang 2024 reicht.

In [ ]:
if BRONZE:
    spaet, alle, frueh, spaetest = con.execute(f'''
        SELECT count(*) FILTER (
                   WHERE epoch_ms(attributes.updated_at)
                       > epoch_ms(attributes.incident_occurred_at) + INTERVAL 1 YEAR),
               count(*),
               min(epoch_ms(attributes.incident_occurred_at)),
               max(epoch_ms(attributes.incident_occurred_at))
        FROM {BRONZE}''').fetchone()
    print(f'Zeilen im Bronze-Jahrgang            : {alle:,}')
    print(f'davon >1 Jahr nach der Tat geaendert : {spaet:,} = {spaet / alle:.1%}')
    print(f'frueheste Tatzeit (UTC)              : {frueh}')
    print(f'spaeteste Tatzeit (UTC)              : {spaetest}')

    leer = con.execute(f'''
        SELECT count(*) FILTER (WHERE attributes.latitude IS NULL
                                   OR attributes.longitude IS NULL),
               count(*) FILTER (WHERE attributes.latitude = 0 AND attributes.longitude = 0)
        FROM {BRONZE}''').fetchone()
    print(f'ohne Koordinate                      : {leer[0]:,}')
    print(f'mit Nullinsel (0/0)                  : {leer[1]:,}')

**Über die Hälfte des Jahrgangs wurde mehr als ein Jahr nach der Tat noch
einmal angefasst.** Die Änderungsdaten verteilen sich dabei über viele Tage,
sind also keine einzelne Sammelaktualisierung — ein Teil davon ist echte
Nachbearbeitung, ein Teil geht auf Neuaufbauten des Mart-Modells zurück. Welcher
Anteil welcher ist, sagt die Quelle nicht.

Fachlich zählt die Folgerung, und die ist eindeutig: **Ein Jahrgang ist nie
fertig.** Deshalb führt jeder Lauf ein Manifest mit Zeitpunkt und Sollmenge, und
deshalb ist ein Lauf unveränderlich, statt an Ort und Stelle überschrieben zu
werden. Ein Vergleich zweier Läufe zeigt dann, was sich geändert hat, statt es
zu verdecken.

**260 Zeilen ohne Koordinate** sind 0,3 % — sie fallen aus der räumlichen
Auswertung, nicht aus der Zählung. Eine Nullinsel bei `0/0` gibt es nicht; das
ist bei Koordinatenquellen keine Selbstverständlichkeit.

## 9. Entscheidungen für die Detroit-Ingestion

| Entscheidung | Begründung |
|---|---|
| Nur Standardbibliothek für den Download | Die Extraktion soll in einer schlanken Laufzeit laufen; `pyarrow` erst beim Schreiben |
| Object-IDs zuerst, Zeilen danach | Der Layer meldet kein `supportsPagination`; die Sollmenge steht damit vor dem ersten Datenabruf fest |
| Blöcke als POST | 2000 IDs in einer URL sprengen Proxy-Limits |
| Zeitfenster in Ortszeit gedacht, in UTC gestellt | Der Layer deklariert UTC; alles andere verschiebt den Jahrgang (Abschnitt 6) |
| Bronze als Parquet in Laufverzeichnissen | Ein Lauf ist unveränderlich; Messung in [`31_storage_evaluation.ipynb`](31_storage_evaluation.ipynb) |
| Rohantwort als `_feature_json` mitgeführt | Die Antwort der Quelle bleibt unverändert nachlesbar, auch wenn die Deutung sich ändert |
| `state_offense_code` als Deliktschlüssel | Er trägt die MICR-Codes, die der Crosswalk kennt (Abschnitt 7) |
| Koordinaten selbst verschnitten | Sonst wäre Detroit anders verortet als Chicago |

Die Umsetzung führt [`12_detroit_bronze_ingestion.ipynb`](12_detroit_bronze_ingestion.ipynb)
vor; der Betrieb steht in [`DETROIT_PIPELINE_ANLEITUNG.md`](../docs/DETROIT_PIPELINE_ANLEITUNG.md).

## 10. Eine einfache Testabfrage

Fünf Zeilen, frisch vom Dienst — dasselbe, was
[`01_chicago_source.ipynb`](01_chicago_source.ipynb) für Chicago tut. Es geht
nicht um die Daten, sondern darum, dass der Zugriff ohne Schlüssel und ohne
Paket funktioniert.

In [ ]:
if NETZ:
    import pandas as pd

    felder = ['incident_entry_id', 'incident_occurred_at', 'offense_category',
              'offense_description', 'state_offense_code', 'arrest_charge',
              'neighborhood', 'latitude', 'longitude']
    antwort = arcgis('/query',
                     where=JAHR_2024,
                     outFields=','.join(felder),
                     orderByFields='incident_occurred_at,incident_entry_id',
                     resultRecordCount=5)
    probe = pd.DataFrame([m['attributes'] for m in antwort['features']], columns=felder)
    probe['incident_occurred_at'] = pd.to_datetime(
        probe['incident_occurred_at'], unit='ms', utc=True)
    display(probe)

## 11. Offizielle Quellen

| Was | Adresse |
| --- | --- |
| Open Data Portal der Stadt Detroit | <https://data.detroitmi.gov/> |
| Datensatz *RMS Crime Incidents* | <https://data.detroitmi.gov/datasets/rms-crime-incidents> |
| Der Feature Layer selbst | <https://services2.arcgis.com/qvkbeam7Wirps6zC/arcgis/rest/services/RMS_Crime_Incidents/FeatureServer/0> |
| ArcGIS REST API, Query-Operation | <https://developers.arcgis.com/rest/services-reference/enterprise/query-feature-service-layer/> |
| MICR beim Michigan State Police | <https://www.michigan.gov/msp/divisions/cjic/micr> |

**Im Repository**

| Was | Wo |
| --- | --- |
| Client und Transformation | [`src/crime/detroit_pipeline/detroit_arcgis.py`](../src/crime/detroit_pipeline/detroit_arcgis.py) |
| Extraktion nach Bronze | [`src/crime/detroit_pipeline/extract_detroit.py`](../src/crime/detroit_pipeline/extract_detroit.py) |
| MICR nach NIBRS | `config/micr_nibrs_mapping.csv`, Quellen `docs/micr_manual_1110_2024.pdf` und `docs/micr_appendix_f_2009.pdf` |
| Die Deliktsprache im Vergleich | [`32_nibrs_source.ipynb`](32_nibrs_source.ipynb) |
| Der Zeitzonen-Befund im Abgabetext | [`docs/ln1_konzept.md`](../docs/ln1_konzept.md), Kapitel 8.6 |

**Zu `michigan.gov`:** Die Seite beantwortet automatisierte Abrufe mit HTTP 403.
Die MICR-Handbücher liegen deshalb als PDF im Repository statt nur als Verweis.

**Stand der Abrufe:** 11. September 2026. Alle Zahlen oben stammen aus den
Zellen dieses Notebooks, nicht aus dieser Tabelle.

In [ ]:
con.close()
print('Verbindung geschlossen.')